In [1]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import History

I0000 00:00:1776116972.129744   45972 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776116972.131330   45972 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776116972.202189   45972 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776116974.055389   45972 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
X_train: tf.data.Dataset = tf.keras.utils.image_dataset_from_directory(
    "data/Train",
    labels="inferred",
    label_mode="int",
    image_size=(600,450),
    batch_size=32
)

X_test: tf.data.Dataset = tf.keras.utils.image_dataset_from_directory(
    "data/Test",
    labels="inferred",
    label_mode="int",
    image_size=(600,450),
    batch_size=32,
    shuffle=False
)

Found 2239 files belonging to 9 classes.
Found 118 files belonging to 9 classes.


E0000 00:00:1776116987.744202   45972 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1776116987.744645   46179 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1776116987.764626   45972 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
X_train_shape = None
for images,_ in X_train.take(1):
    X_train_shape = images.shape

In [4]:
model = tf.keras.Sequential([
    layers.Input(shape=X_train_shape[1:]),
    layers.Rescaling(1./255),

    layers.Conv2D(32,3,activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64,3,activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Flatten(),
    layers.Dense(64,activation='relu'),
    layers.Dense(9,activation='softmax')
])

In [5]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # SCCE = -1 * ( \sum_{i=1}^{n} \log(\hat{y}_{i, y_i}) )
    metrics=['accuracy']
)

In [6]:
history: History = model.fit(
    X_train,
    epochs=5,
    batch_size=32
)

Epoch 1/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.2564 - loss: 4.6476
Epoch 2/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 130s 2s/step - accuracy: 0.3220 - loss: 1.8224
Epoch 3/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.3658 - loss: 1.6642
Epoch 4/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 130s 2s/step - accuracy: 0.4113 - loss: 1.5744
Epoch 5/5
70/70 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.4676 - loss: 1.4833


In [7]:
evaluation = model.evaluate(X_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 527ms/step - accuracy: 0.3305 - loss: 2.4560
